# SHL Grammar Scoring Engine - Final Submission Notebook

> **How to use.** Run the pipeline scripts first (`audit_dataset`, `transcribe`, `extract_features`, `train`), then *Restart & Run All* here. Every number in this notebook is computed from the real data at execution time. Where an artifact is missing the cell prints: *Results will be populated after local execution.*

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
from src.config import load_config, repo_dir
from src.pipeline import get_context

# Dataset location: leave None to use the SHL_DATA_ROOT environment variable (or config.data_root),
# or set it here, e.g. DATA_ROOT = r"C:\Users\USERNAME\path\Dataset_Final"
DATA_ROOT = None
try:
    ctx = get_context(DATA_ROOT); cfg = ctx.cfg
    print("Data root:", ctx.paths.root)
except Exception as exc:
    ctx, cfg = None, load_config()
    print("Data root NOT configured ->", exc)
PENDING = "Results will be populated after local execution."
TABLES = repo_dir(cfg, "reports_dir") / "tables"
LOG_PATH = repo_dir(cfg, "experiments_dir") / "experiment_log.csv"
def have(*paths): return all(Path(p).exists() for p in paths)
from src.models.train import read_log, select_best, fit_final, get_feature_importance
from src.evaluation.metrics import rmse, pearson, clip_predictions
from src import reporting as rp

## 1. Executive summary

**Task.** Predict a continuous grammar score in [0, 5] from ~45-60 s of spoken English.
**Approach.** Local ASR (faster-whisper) -> acoustic + transcript-derived linguistic features + (optional) MiniLM embeddings -> tree-based regression, selected by 5-fold CV on RMSE (tie-break Pearson), predictions clipped to [0, 5].
**Why this design.** Only ~769 labelled samples, so a compact, explainable feature-based regressor is preferred to training a neural network from scratch; every component is justified by an ablation in the experiment ladder.

In [ ]:
if have(LOG_PATH):
    log = read_log(LOG_PATH)
    summary = log.sort_values("cv_rmse_mean").head(5)[["experiment_id", "feature_set", "model", "cv_rmse_mean", "cv_pearson_mean", "training_rmse"]]
    print("Top candidates by cross-validated RMSE:"); display(summary.round(4))
else:
    print(PENDING)

## 2. Problem statement
Given a spoken-English audio file, output a **continuous** grammar score from 0 to 5. The score follows the SHL rubric:

| Score | Rubric (abridged) |
|---|---|
| 1 | Struggles with sentence structure/syntax; limited control of simple, memorised patterns |
| 2 | Limited understanding; consistent basic errors; may leave sentences incomplete |
| 3 | Decent grasp of one of structure/grammar but errors in the other |
| 4 | Strong, consistent control; occasional minor errors that rarely impede meaning; can self-correct |
| 5 | High accuracy, adept control of complex grammar; seldom makes noticeable mistakes; self-corrects |

**Why regression, not classification:** scores are ordinal and appear at half-point steps in the training labels; the leaderboard metrics (RMSE, Pearson) reward getting *close*, which a 5-class classifier would ignore (predicting 4 for a true 5 is as wrong as predicting 1).

## 3. Assignment requirements checklist
- [x] Jupyter notebook with documented code and a brief report (this notebook)
- [x] Approach, preprocessing, pipeline architecture and evaluation explained
- [x] **Training RMSE** reported (section 18), clearly separated from validation / CV RMSE (section 19)
- [x] RMSE and Pearson correlation computed
- [x] Visualisations
- [x] Predictions in the Kaggle format, validated against `sample_submission.csv`
- [x] No external labelled grammar data, no test labels, no paid APIs

## 4. Dataset overview

In [ ]:
from src.data.loader import load_all_metadata
if ctx is not None and not ctx.paths.missing_csvs():
    meta = load_all_metadata(ctx.paths)
    print(f"train: {len(meta.train)} | test: {len(meta.test)} | sample_submission: {len(meta.sample)}")
    y_all = meta.train["label"].astype(float)
    display(y_all.describe().to_frame().T.round(3))
else:
    meta = None; print(PENDING)

## 5. Data integrity checks

In [ ]:
if meta is not None:
    from src.data.validator import audit_metadata
    audit = audit_metadata(meta.train, meta.test, meta.sample, cfg["expected"]["train_rows"], cfg["expected"]["test_rows"])
    for f in audit.findings: print(f"[{f.level.upper():7s}] {f.code}: {f.message}")
    METADATA_OK = not audit.has_errors
    print("\nMetadata consistent:", METADATA_OK)

## 6. Exploratory data analysis

In [ ]:
if meta is not None:
    rp.plot_label_distribution(meta.train["label"]); plt.show()
inv_path = TABLES / "audio_inventory.csv"
if have(inv_path):
    inv = pd.read_csv(inv_path); inv = inv[(inv["split"] == "train") & inv["ok"]]
    rp.plot_hist(inv["duration"], "Audio duration (train)", "Seconds", name="duration_distribution"); plt.show()
    rp.plot_value_counts(inv["sample_rate"].astype(int), "Sample rate (train)", "Hz", name="sample_rate_distribution"); plt.show()
else:
    print("Run scripts/audit_dataset.py for duration / sample-rate plots.", PENDING)

## 7. Audio preprocessing
`src/audio/preprocessing.py`: decode -> mono -> 16 kHz resample -> DC removal -> (feature-specific) safe peak normalisation. Corrupt, empty or too-short files are detected and recorded, never silently dropped. **Original WAV files are never modified.**

In [ ]:
if meta is not None and ctx.paths.train_audio_dir.is_dir():
    from src.audio.preprocessing import load_audio
    r = meta.train.iloc[0]
    y_, sr_ = load_audio(ctx.paths.train_audio_dir / r.filename, sr=cfg["audio"]["sample_rate"])
    rp.plot_waveform_spectrogram(y_, sr_, title=f"{r.filename} (label {r.label})"); plt.show()
else:
    print(PENDING)

## 8. ASR
`faster-whisper` (`small.en`, fallback `base.en`) runs locally on GPU if present, else CPU (int8). Chosen because it is open-source, free, offline, fast enough for ~1,000 clips and English-specialised models are more accurate than multilingual ones of similar size. Transcripts are cached and the run is resumable (`scripts/transcribe.py`).

In [ ]:
tp = ctx.processed("transcripts") / "train_transcripts.csv" if ctx else None
if tp is not None and tp.exists():
    tr = pd.read_csv(tp); tr["transcript"] = tr["transcript"].fillna("")
    print("status counts:", tr["status"].value_counts().to_dict(), "| models:", tr["asr_model"].unique().tolist())
else:
    tr = None; print(PENDING)

## 9. Transcript analysis

In [ ]:
if tr is not None and meta is not None:
    t = tr.merge(meta.train, on="filename"); t["n_words"] = t["transcript"].str.split().str.len()
    display(t.groupby("label")["n_words"].agg(["count", "mean"]).round(1))
    print("empty transcripts:", int((t["transcript"].str.strip() == "").sum()))

## 10. Acoustic features (`aud_`)
Duration; RMS and zero-crossing statistics; spectral centroid / bandwidth / rolloff / flatness / contrast; 13 MFCCs plus delta statistics; speech/silence ratio; pause count, mean, max and total (pauses >= 0.3 s); onset-rate (speaking-rate proxy); rough YIN pitch statistics. They capture fluency and delivery, which correlate with proficiency but are **not direct grammar measurements**.

## 11. Linguistic features (`txt_`, `ling_`)
Word/sentence counts and lengths, vocabulary size and type-token ratios, repeated-word ratio, filler counts, punctuation statistics; function-word ratios (pronoun, auxiliary/modal, conjunction, preposition, determiner), subordination/relative-clause markers, passive/perfect/conditional patterns, simple agreement and article rule flags, short/long/unterminated sentence ratios. These are **lexicon/rule-based proxies** for grammatical control, not a grammar checker.

## 12. Text embeddings (`emb_`)
`sentence-transformers/all-MiniLM-L6-v2` (384-d, local, no API). With only ~769 labels, embeddings are reduced with **PCA fitted inside every training fold**. They are optional: EXP-03/05/05b compare handcrafted text, embeddings, and both.

## 13. Feature fusion

In [ ]:
train_path = ctx.processed("merged_features") / "train_features.csv" if ctx else None
READY = bool(ctx) and have(train_path, LOG_PATH)
if READY:
    from src.pipeline import load_train_matrix
    X, y = load_train_matrix(ctx)
    from src.features.fusion import group_columns
    print("training matrix:", X.shape, {g: len(group_columns(X.columns, [g])) for g in ("audio", "text", "embeddings")})
else:
    print(PENDING)

## 14. Model candidates
Mean / median baselines; Ridge (on embeddings); ExtraTrees; RandomForest; HistGradientBoosting; LightGBM; XGBoost. Hyper-parameters are deliberately modest and fixed (shallow trees, subsampling, regularisation) because heavy tuning on ~769 samples mostly fits noise. Trees handle mixed-scale tabular features, missing values and non-linear interactions without scaling.

## 15. Cross-validation strategy
5-fold KFold, **stratified on equal-frequency bins of the score**, with a fixed seed so every candidate sees identical folds. For every fold: imputer, PCA and the model are fitted on the training portion only; predictions on the held-out fold are clipped to [0, 5]; RMSE and Pearson are computed per fold and summarised as mean +/- std. The test set plays no role in model selection.

In [ ]:
if READY:
    from src.evaluation.cross_validation import make_folds
    folds = make_folds(y, cfg["validation"]["folds"], cfg["random_seed"], cfg["validation"]["stratify_bins"])
    print(pd.DataFrame([{"fold": i, "train": len(a), "validation": len(b), "val_mean_label": round(float(y[b].mean()), 3)} for i, (a, b) in enumerate(folds)]))

## 16. Experiment results

In [ ]:
if READY:
    log = read_log(LOG_PATH)
    display(log.sort_values("cv_rmse_mean")[["experiment_id", "feature_set", "model", "cv_rmse_mean", "cv_rmse_std", "cv_pearson_mean", "cv_pearson_std", "training_rmse"]].round(4))
    rp.plot_model_comparison(log); plt.show()

## 17. Final model selection
Rule (fixed in advance): the candidate with the **lowest mean CV RMSE** wins; ties are broken by higher mean CV Pearson. Baselines are excluded from selection but reported. The ensemble competes as a normal candidate, so it is used only if CV shows it is better.

In [ ]:
if READY:
    best = select_best(log)
    print(f"Selected: {best['experiment_id']} | {best['model']} | features = {best['feature_set']}")
    display(best.to_frame("value"))
    runner_up = log[~log.model.isin(["mean_baseline", "median_baseline"])].sort_values("cv_rmse_mean").iloc[1]
    gap = runner_up["cv_rmse_mean"] - best["cv_rmse_mean"]
    print(f"Margin over runner-up ({runner_up['experiment_id']} {runner_up['model']}): {gap:.4f} RMSE; fold std of winner = {best['cv_rmse_std']:.4f}")
    if gap < best["cv_rmse_std"]:
        print("NOTE: margin is smaller than fold-to-fold std - prefer the simpler candidate if it is within noise.")

## 18. Training RMSE (in-sample)
> **Training RMSE is NOT a generalisation estimate.** It is computed by predicting the same rows the final model was fitted on, so it is optimistic - especially for tree ensembles. It is reported because the assignment requires it; judge the model by the cross-validated numbers in section 19.

In [ ]:
if READY:
    final_model, train_rmse = fit_final(best, X, y, cfg)
    train_pred = clip_predictions(final_model.predict(X), cfg["prediction"]["min_score"], cfg["prediction"]["max_score"])
    print(f"TRAINING RMSE (in-sample, all {len(y)} labelled rows): {rmse(y, train_pred):.4f}")
    print(f"Training Pearson (in-sample):                         {pearson(y, train_pred):.4f}")

## 19. Validation (cross-validated) RMSE
Out-of-fold predictions: every training row is predicted by a model that never saw it.

In [ ]:
if READY:
    oof = pd.read_csv(TABLES / "oof_predictions.csv")
    col = f"{best['experiment_id']}__{best['model']}"
    members_ok = col in oof.columns
    if not members_ok and str(best["model"]).startswith("ensemble"):
        import json
        mem = [f"{m['experiment_id']}__{m['model']}" for m in json.loads(best["hyperparameters"])["members"]]
        oof[col] = oof[mem].mean(axis=1); members_ok = True
    print(f"Mean CV RMSE   : {best['cv_rmse_mean']:.4f} +/- {best['cv_rmse_std']:.4f}   (5 folds)")
    print(f"Training RMSE  : {train_rmse:.4f}   <- in-sample, for comparison only")
    print(f"Optimism gap   : {best['cv_rmse_mean'] - train_rmse:.4f}")
    base = log[log.model == "mean_baseline"]
    if len(base): print(f"Mean-baseline CV RMSE: {base.iloc[0]['cv_rmse_mean']:.4f}")
    if members_ok:
        print(f"Pooled out-of-fold RMSE: {rmse(oof['y_true'], oof[col]):.4f}")

## 20. Pearson correlation

In [ ]:
if READY and members_ok:
    print(f"Mean CV Pearson : {best['cv_pearson_mean']:.4f} +/- {best['cv_pearson_std']:.4f}")
    print(f"Pooled OOF Pearson: {pearson(oof['y_true'], oof[col]):.4f}")
    fm = rp.fold_metrics_from_oof(oof, [col]); display(fm.round(4))

## 21. Error analysis (out-of-fold predictions on the labelled training data)

In [ ]:
if READY and members_ok:
    yt, yp = oof["y_true"].to_numpy(), oof[col].to_numpy()
    tbl = rp.error_by_range(yt, yp); display(tbl.round(3))
    print("over-predicted rows :", int((yp > yt).sum()), "| under-predicted rows:", int((yp < yt).sum()))
    worst = rp.worst_predictions(oof["filename"], yt, yp, 15); display(worst.round(3))
    if tr is not None:
        display(worst.merge(tr[["filename", "transcript"]], on="filename", how="left").assign(transcript=lambda d: d.transcript.str[:200])[["filename", "y_true", "y_pred", "transcript"]].head(8))

## 22. Visualisations

In [ ]:
if READY and members_ok:
    rp.plot_actual_vs_pred(yt, yp); plt.show()
    rp.plot_residuals(yt, yp); plt.show()
    rp.plot_prediction_distribution(yt, yp); plt.show()
    rp.plot_error_by_range(tbl); plt.show()
    cands = log[~log.model.isin(["mean_baseline", "median_baseline"]) & ~log.model.str.startswith("ensemble")].sort_values("cv_rmse_mean").head(4)
    cols = [f"{a}__{b}" for a, b in zip(cands.experiment_id, cands.model) if f"{a}__{b}" in oof.columns]
    rp.plot_cv_folds(rp.fold_metrics_from_oof(oof, cols)); plt.show()
    imp = get_feature_importance(final_model)
    if imp is not None:
        rp.plot_feature_importance(imp); plt.show(); display(imp.head(15).round(4))
    else:
        print("Feature importances are available for single tree models (not ensembles/ridge).")

## 23. Final test inference
Test features are built from the cached transcripts / acoustic features of the test audio. Predictions are clipped to [0, 5]. Test labels (-1 in `test.csv`) are never read as targets.

In [ ]:
if READY:
    from src.pipeline import build_merged
    from src.models.predict import predict_scores
    MANIFEST = "test"   # use "sample_submission" only if you decided sample_submission.csv defines the scored files
    try:
        test_feats = build_merged(ctx, "test", MANIFEST, strict=True)
        test_pred = predict_scores(final_model, test_feats, cfg["prediction"]["min_score"], cfg["prediction"]["max_score"])
        display(test_pred.describe().T.round(3))
        rp.plot_hist(test_pred["label"], "Predicted test scores", "Score", name="test_prediction_distribution", bins=25); plt.show()
    except Exception as exc:
        test_pred = None; print("Test inference blocked:", exc)

## 24. Submission validation
The submission is checked against the official `sample_submission.csv` (columns and order, duplicates, NaNs, row count, filename set, range). If the metadata is inconsistent the notebook **stops and reports it** instead of producing a valid-looking file.

In [ ]:
if READY and test_pred is not None:
    from src.data.validator import assert_test_matches_sample, MetadataInconsistencyError
    from src.submission import validate_submission, write_submission, submission_path, SubmissionError
    try:
        if set(test_pred["filename"]) != set(meta.sample["filename"]):
            assert_test_matches_sample(meta.test, meta.sample)
        valid = validate_submission(test_pred, meta.sample, cfg["prediction"]["min_score"], cfg["prediction"]["max_score"])
        out = write_submission(valid, submission_path(repo_dir(cfg, "submissions_dir"), None, "final"),
                               protected=ctx.paths.sample_submission_csv, overwrite=True)
        print("VALID. Submission written to:", out); display(valid.head())
    except (MetadataInconsistencyError, SubmissionError) as exc:
        print("NO SUBMISSION WRITTEN:\n", exc)

## 25. Limitations
* ASR may normalise or "repair" ungrammatical speech, hiding errors from the text features.
* Linguistic features are lexicon/rule proxies, not a parser; they detect few real grammar errors.
* ~769 labelled samples: CV estimates have noticeable variance (see fold std); small differences between candidates are not significant.
* Labels come from human raters on a coarse rubric; rater noise bounds the achievable RMSE.
* Acoustic features measure fluency/delivery, which correlates with but is not grammar.
* Accent / recording-quality shift between train and test is not modelled.
* Metadata inconsistency between `test.csv` and `sample_submission.csv` (see section 5) must be resolved with the organisers.

## 26. Future improvements
* Grammar-aware features from a parser or grammatical-error-correction model (edit distance between ASR text and its corrected version) - open-source models only.
* Fine-tune a small speech/text encoder with the ordinal rubric once more labelled data exists.
* Ordinal-aware losses / quantile calibration of the predicted distribution.
* Repeated CV and paired fold tests for model comparison; stacking with a ridge meta-learner.
* Confidence-weighted ASR features (word-level probabilities) to detect disfluent regions.

## 27. Reproducibility

In [ ]:
import platform, sklearn, json
print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)
for pkg in ("lightgbm", "xgboost", "librosa", "faster_whisper", "sentence_transformers"):
    try: print(pkg, __import__(pkg).__version__)
    except Exception as e: print(pkg, "unavailable")
print("random_seed:", cfg["random_seed"], "| folds:", cfg["validation"]["folds"], "| ASR:", cfg["asr"]["model"], "| embedder:", cfg["features"]["embedding_model"])
print("Pipeline: scripts/audit_dataset.py -> transcribe.py -> extract_features.py -> train.py --fit-final -> predict.py -> make_submission.py")